# Qwen 3.5 4B через llama.cpp

> Сборка llama.cpp и модель берутся из каталогов текущего ZEMI Instance через `zemi.env.path`.

## Запуск модели

In [7]:
from zemi import exp

llama_config = {
    "llama_build": "b10202",
    "model_owner": "bartowski",
    "model_repository": "Llama-3.2-3B-Instruct-uncensored-GGUF",
    "model_filename": "Llama-3.2-3B-Instruct-uncensored-Q5_K_M.gguf",
}

exp.llama.download(**llama_config)

llama.cpp b10202 уже загружен: C:\Users\Axoman\Documents\ZEMI\_llamas\llama--b10202
Скачиваю модель bartowski/Llama-3.2-3B-Instruct-uncensored-GGUF/Llama-3.2-3B-Instruct-uncensored-Q5_K_M.gguf...


'Модель bartowski/Llama-3.2-3B-Instruct-uncensored-GGUF/Llama-3.2-3B-Instruct-uncensored-Q5_K_M.gguf: 100.0% · 2.4 ГБ / 2.4 ГБ · 11.4 МБ/с · готово'

Модель загружена: C:\Users\Axoman\Documents\ZEMI\_models\hf--bartowski--Llama-3.2-3B-Instruct-uncensored-GGUF--Llama-3.2-3B-Instruct-uncensored-Q5_K_M\Llama-3.2-3B-Instruct-uncensored-Q5_K_M.gguf


(WindowsPath('C:/Users/Axoman/Documents/ZEMI/_llamas/llama--b10202'),
 WindowsPath('C:/Users/Axoman/Documents/ZEMI/_models/hf--bartowski--Llama-3.2-3B-Instruct-uncensored-GGUF--Llama-3.2-3B-Instruct-uncensored-Q5_K_M/Llama-3.2-3B-Instruct-uncensored-Q5_K_M.gguf'))

In [ ]:
exp.llama.restart(**llama_config, alias="Llama-3.2-3B")

Работающий llama-server не найден


RuntimeError: llama-server завершился с кодом 1

## Диалог с моделью

In [ ]:
from pathlib import Path
from openai import OpenAI


client = OpenAI(
    base_url="http://127.0.0.1:8080/v1",
    api_key="not-needed",
    timeout=300.0,
)

# Если ноутбук и файлы лежат в одной папке, достаточно относительных путей.
# Для общей папки VMware можно указать, например:
# DATA_DIR = Path(r"Z:\VMShare")
DATA_DIR = Path.cwd() / "data/case01"

EXCEL_FILES = [
    DATA_DIR / "Отчет 1.xlsx",
    DATA_DIR / "Отчет 2.xlsx",
    DATA_DIR / "Отчет 3.xlsx",
]


excel_context = "\n\n".join(exp.excel.excel_to_text(path) for path in EXCEL_FILES)

print("Загружены файлы:")
for path in EXCEL_FILES:
    print(f"- {path.name}: {path.stat().st_size} байт")

print(f"\nРазмер подготовленного текста: {len(excel_context)} символов")

TASK = """
Обработай все переданные Excel-отчёты.

Для каждого файла извлеки:
- название города/филиала;
- дату выгрузки;
- руководителя филиала;
- строки таблицы с полями date, article, cost.

Не включай строку «Итого».
Верни только корректный JSON без Markdown и пояснений.
Формат результата:
{
  "reports": [
    {
      "source_file": "имя файла",
      "city": "город",
      "export_date": "YYYY-MM-DD",
      "manager": "ФИО",
      "transactions": [
        {"date": "YYYY-MM-DD", "article": "...", "cost": 0}
      ]
    }
  ]
}
""".strip()

print("\nОтправляю три Excel-файла модели...")

response = client.chat.completions.create(
    model="qwen3.5-4b",
    messages=[
        {
            "role": "system",
            "content": (
                "Ты аккуратно преобразуешь содержимое Excel-отчётов "
                "в строго структурированный JSON. Не выдумывай отсутствующие данные."
            ),
        },
        {
            "role": "user",
            "content": f"{TASK}\n\nДАННЫЕ EXCEL:\n{excel_context}",
        },
    ],
    temperature=0.0,
    max_tokens=2048,
)

text = response.choices[0].message.content

if text is None:
    raise RuntimeError("Модель вернула пустой ответ")

print("\nОтвет модели:\n")
print(text)


Загружены файлы:
- Отчет 1.xlsx: 10365 байт
- Отчет 2.xlsx: 10291 байт
- Отчет 3.xlsx: 10530 байт

Размер подготовленного текста: 962 символов

Отправляю три Excel-файла модели...

Ответ модели:

```json
{
  "reports": [
    {
      "source_file": "Отчет 1.xlsx",
      "city": "Москва",
      "export_date": "2026-04-30",
      "manager": "Иванов И.И.",
      "transactions": [
        {"date": "2026-03-14", "article": "15", "cost": 83204},
        {"date": "2026-03-24", "article": "61", "cost": 53307},
        {"date": "2026-03-09", "article": "56", "cost": 67750},
        {"date": "2026-05-17", "article": "65", "cost": 21224},
        {"date": "2026-04-21", "article": "78", "cost": 65206},
        {"date": "2026-03-27", "article": "61", "cost": 40616},
        {"date": "2026-02-14", "article": "14", "cost": 33626}
      ]
    },
    {
      "source_file": "Отчет 2.xlsx",
      "city": "Санкт-Петербург",
      "export_date": "2026-04-30",
      "manager": "Петров П.П.",
      "transacti

In [ ]:
exp.chat.print_response(response)

```json
{
  "reports": [
    {
      "source_file": "Отчет 1.xlsx",
      "city": "Москва",
      "export_date": "2026-04-30",
      "manager": "Иванов И.И.",
      "transactions": [
        {"date": "2026-03-14", "article": "15", "cost": 83204},
        {"date": "2026-03-24", "article": "61", "cost": 53307},
        {"date": "2026-03-09", "article": "56", "cost": 67750},
        {"date": "2026-05-17", "article": "65", "cost": 21224},
        {"date": "2026-04-21", "article": "78", "cost": 65206},
        {"date": "2026-03-27", "article": "61", "cost": 40616},
        {"date": "2026-02-14", "article": "14", "cost": 33626}
      ]
    },
    {
      "source_file": "Отчет 2.xlsx",
      "city": "Санкт-Петербург",
      "export_date": "2026-04-30",
      "manager": "Петров П.П.",
      "transactions": [
        {"date": "2026-01-27", "article": "21", "cost": 3515},
        {"date": "2026-05-11", "article": "48", "cost": 1969},
        {"date": "2026-02-18", "article": "33", "cost": 2964},

'```json\n{\n  "reports": [\n    {\n      "source_file": "Отчет 1.xlsx",\n      "city": "Москва",\n      "export_date": "2026-04-30",\n      "manager": "Иванов И.И.",\n      "transactions": [\n        {"date": "2026-03-14", "article": "15", "cost": 83204},\n        {"date": "2026-03-24", "article": "61", "cost": 53307},\n        {"date": "2026-03-09", "article": "56", "cost": 67750},\n        {"date": "2026-05-17", "article": "65", "cost": 21224},\n        {"date": "2026-04-21", "article": "78", "cost": 65206},\n        {"date": "2026-03-27", "article": "61", "cost": 40616},\n        {"date": "2026-02-14", "article": "14", "cost": 33626}\n      ]\n    },\n    {\n      "source_file": "Отчет 2.xlsx",\n      "city": "Санкт-Петербург",\n      "export_date": "2026-04-30",\n      "manager": "Петров П.П.",\n      "transactions": [\n        {"date": "2026-01-27", "article": "21", "cost": 3515},\n        {"date": "2026-05-11", "article": "48", "cost": 1969},\n        {"date": "2026-02-18", "art

## Остановка модели

Выполни эту ячейку, когда модель больше не нужна.

In [ ]:
exp.llama.stop()

llama-server остановлен, PID: 12536


True